# Qwen3-4B LoRA: 5 CL-запусков по `pythoncodes_cl_scored`

Цель ноутбука: нормально и воспроизводимо запустить пять curriculum-learning вариантов LoRA для `pythoncodes`:

1. `cl_length`
2. `cl_perplexity`
3. `cl_lexical`
4. `cl_semantic`
5. `cl_llm_judge`

Ноутбук не содержит «магии обучения» внутри ячеек. Он управляет уже вынесенным кодом из `src.train.lora_train`: проверяет окружение, проверяет датасет, выбирает ровно нужные CL-запуски, запускает обучение, собирает train/eval-логи в pandas-таблицы и сохраняет их в `outputs/`.


## 0. Главные переключатели

Перед запуском на кластере обычно достаточно поменять только эти значения.

`RUN_MODE = "full"` запускает все 5 CL-экспериментов. Для короткой проверки поставь `"smoke"` — тогда будет один маленький запуск на 10 шагов.

`SELECTED_MODEL = "4b-thinking"` оставлен как в старом ноутбуке. Если хочешь именно instruct-вариант из `src/config.py`, поставь `"4b-instruct"`.


In [1]:
import os
from pathlib import Path

SELECTED_MODEL = "4b-instruct"   # "4b-thinking" или "4b-instruct" из src/config.py
MAX_TOKENS = 4096

RUN_MODE = "smoke"                # "full" или "smoke"
FORCE_RETRAIN = False            # False: пропускать уже завершённые запуски
INCLUDE_BASELINE_IN_EVAL = True # True: добавить base model в eval-таблицу

# Для compute-node на кластере обычно нужен offline-режим.
# Для login-node при докачке модели поставь OFFLINE_MODE = False.
OFFLINE_MODE = True

# Для smoke eval поставь маленькое число. Для полного eval поставь None.
EVAL_MAX_TASKS = 5 if RUN_MODE == "smoke" else None

CL_RUN_NAMES = [
    "cl_length",
    "cl_perplexity",
    "cl_lexical",
    "cl_semantic",
    "cl_llm_judge",
]

os.environ["SELECTED_MODEL"] = SELECTED_MODEL
os.environ["MAX_TOKENS"] = str(MAX_TOKENS)

if OFFLINE_MODE:
    os.environ.setdefault("HF_HUB_OFFLINE", "1")
    os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")


## 1. Найти корень репозитория и импортировать проект

Старый ноутбук ломался, если запускать его из папки `notebooks/`, потому что `Path.cwd()` мог указывать не на корень проекта. Здесь корень ищется по наличию `src/` и `configs/`.


In [2]:
import sys
import json
import time
import shutil
from copy import deepcopy
from pathlib import Path

import pandas as pd

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "src").exists() and (path / "configs").exists():
            return path
    raise RuntimeError(
        "Не нашёл корень проекта. Запусти ноутбук из master-cluster-HSE "
        "или положи его внутрь репозитория."
    )

PROJECT_DIR = find_project_root()
os.chdir(PROJECT_DIR)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# Важно: env-переменные выше должны быть выставлены ДО импорта src.config.
import src.config as config

from src.train.lora_train import (
    load_train_config,
    load_pythoncodes_cl_df,
    train_lora_experiments,
    collect_training_table,
    evaluate_lora_experiments,
)

print("PROJECT_DIR:", PROJECT_DIR)
print("SELECTED_MODEL:", config.SELECTED_MODEL)
print("MODEL_NAME:", config.MODEL_NAME)
print("MODEL_PATH:", config.MODEL_PATH)
print("DATASETS_DIR:", config.DATASETS_DIR)
print("OUTPUTS_DIR:", config.OUTPUTS_DIR)
print("MODELS_DIR:", config.MODELS_DIR)
print("MAX_TOKENS:", config.MAX_TOKENS)
print("OFFLINE_MODE:", OFFLINE_MODE)


/home/pavel/projects/mouse-learning/master-cluster-HSE/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/pavel/projects/mouse-learning/master-cluster-HSE/src/train/lora_train.py:15: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


[bitsandbytes.cextension|ERROR]bitsandbytes library load error: libnvJitLink.so.13: cannot open shared object file: No such file or directory
Traceback (most recent call last):
  File "/home/pavel/projects/mouse-learning/master-cluster-HSE/.venv/lib/python3.12/site-packages/bitsandbytes/cextension.py", line 320, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/home/pavel/projects/mouse-learning/master-cluster-HSE/.venv/lib/python3.12/site-packages/bitsandbytes/cextension.py", line 298, in get_native_library
    dll = ct.cdll.LoadLibrary(str(binary_path))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 460, in LoadLibrary
    return self._dlltype(name)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: libnvJitLink.so.13: cannot open shared object file:

INFO 05-01 03:13:18 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 05-01 03:13:18 [nixl_utils.py:34] NIXL is not available
WARNING 05-01 03:13:18 [nixl_utils.py:44] NIXL agent config is not available
WARNING 05-01 03:13:19 [interface.py:686] Using 'pin_memory=False' as WSL is detected. This may slow down the performance.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
PROJECT_DIR: /home/pavel/projects/mouse-learning/master-cluster-HSE
SELECTED_MODEL: 4b-instruct
MODEL_NAME: Qwen/Qwen3-4B-Instruct-2507
MODEL_PATH: /home/pavel/projects/mouse-learning/master-cluster-HSE/models/qwen3-4b-instruct-2507
DATASETS_DIR: /home/pavel/projects/mouse-learning/master-cluster-HSE/datasets
OUTPUTS_DIR: /home/pavel/projects/mouse-learning/master-cluster-HSE/outputs
MODELS_DIR: /home/pavel

## 2. Проверить конфиг обучения

Ожидается файл `configs/train/lora_pythoncodes_cl.yaml`. Из него берутся LoRA-параметры, batch size, gradient accumulation, max steps, eval/logging steps и список curriculum-запусков.

Ноутбук дальше принудительно выбирает только пять CL-запусков из `CL_RUN_NAMES`, без обычного `sft_pythoncodes`.


In [3]:
CONFIG_PATH = PROJECT_DIR / "configs" / "train" / "lora_pythoncodes_cl.yaml"

train_cfg = load_train_config(str(CONFIG_PATH))
runs_by_name = {run["name"]: run for run in train_cfg["runs"]}

missing_runs = [name for name in CL_RUN_NAMES if name not in runs_by_name]
if missing_runs:
    raise ValueError(f"В конфиге нет запусков: {missing_runs}")

cl_runs = [runs_by_name[name] for name in CL_RUN_NAMES]

print("CONFIG_PATH:", CONFIG_PATH)
display(pd.DataFrame(cl_runs))

print("training:")
display(pd.Series(train_cfg["training"]))

print("lora:")
display(pd.Series(train_cfg["lora"]))


CONFIG_PATH: /home/pavel/projects/mouse-learning/master-cluster-HSE/configs/train/lora_pythoncodes_cl.yaml


,name,category_col
0,cl_length,length_category
1,cl_perplexity,ppl_category
2,cl_lexical,lexical_cluster_category
3,cl_semantic,semantic_cluster_category
4,cl_llm_judge,llm_judge_category


training:


train_batch_size                    1
gradient_steps                      8
max_steps                         600
stage_max_steps                   200
warmup_steps                       25
lr                             0.0002
lr_scheduler_type              cosine
optim                      adamw_8bit
assistant_only_loss              True
packing                         False
logging_steps                      10
eval_steps                         50
save_steps                        100
save_total_limit                    2
report_to                 tensorboard
dataloader_num_workers              2
dataloader_pin_memory            True
max_grad_norm                     1.0
dtype: object

lora:


r                                                                            32
alpha                                                                        32
dropout                                                                       0
bias                                                                       none
use_gradient_checkpointing                                              unsloth
target_modules                [q_proj, k_proj, v_proj, o_proj, gate_proj, up...
dtype: object

## 3. Проверить модель на диске

На compute-node интернета обычно нет, поэтому эта ячейка не скачивает модель. Она только показывает, существует ли локальная папка модели.

Если модель ещё не загружена, запускай следующую download-ячейку на login-node с интернетом и `OFFLINE_MODE = False`.


In [4]:
model_path = Path(config.MODEL_PATH)

print("MODEL_NAME:", config.MODEL_NAME)
print("MODEL_PATH:", model_path)
print("exists:", model_path.exists())

if model_path.exists():
    files = sorted(p.name for p in model_path.iterdir())[:20]
    print("first files:", files)
else:
    print("Модель не найдена локально. Нужно заранее скачать её в MODEL_PATH.")


MODEL_NAME: Qwen/Qwen3-4B-Instruct-2507
MODEL_PATH: /home/pavel/projects/mouse-learning/master-cluster-HSE/models/qwen3-4b-instruct-2507
exists: True
first files: ['.cache', '.gitattributes', 'LICENSE', 'README.md', 'config.json', 'generation_config.json', 'merges.txt', 'model-00001-of-00003.safetensors', 'model-00002-of-00003.safetensors', 'model-00003-of-00003.safetensors', 'model.safetensors.index.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']


### Опционально: скачать модель на login-node

Не запускай эту ячейку на compute-node без интернета. По умолчанию `DO_DOWNLOAD_MODEL = False`.


In [5]:
DO_DOWNLOAD_MODEL = False

if DO_DOWNLOAD_MODEL:
    if OFFLINE_MODE:
        raise RuntimeError("OFFLINE_MODE=True. Для скачивания поставь OFFLINE_MODE=False и перезапусти импорт config.")

    from huggingface_hub import snapshot_download

    snapshot_download(
        repo_id=config.MODEL_NAME,
        local_dir=config.MODEL_PATH,
        local_dir_use_symlinks=False,
        resume_download=True,
    )
else:
    print("Скачивание пропущено.")


Скачивание пропущено.


## 4. Загрузить и проверить `pythoncodes_cl_scored`

Ожидаемый распакованный датасет:

```text
datasets/pythoncodes_cl_scored/
  data-00000-of-00001.arrow
  state.json
  dataset_info.json
```

Если папки нет, код попробует взять parquet из `outputs/curriculum_array/pythoncodes_cl_scored.parquet`, как прописано в YAML.


In [ ]:
df = load_pythoncodes_cl_df(train_cfg)
display(df)

required_category_cols = [
    "length_category",
    "ppl_category",
    "lexical_cluster_category",
    "semantic_cluster_category",
    # "llm_judge_category",
]

missing_cols = [col for col in required_category_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"В датасете нет нужных CL-колонок: {missing_cols}")

print("rows:", len(df))
print("columns:", len(df.columns))
display(df.head(3))

coverage_rows = []
for col in required_category_cols:
    counts = df[col].value_counts(dropna=False).to_dict()
    coverage_rows.append({
        "column": col,
        "non_null": int(df[col].notna().sum()),
        "null": int(df[col].isna().sum()),
        "unique_non_null": sorted(str(x) for x in df[col].dropna().unique()),
        "counts": counts,
    })

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)


,output,instruction,input,text,length_score,lexical_cluster_score,semantic_cluster_score,ppl_score,ifd_score,cond_loss,uncond_loss,length_category,ppl_category,ifd_category,lexical_cluster_category,semantic_cluster_category
0,```python\ntasks = []\nwhile True:\n task =...,Help me set up my daily to-do list!,Setting up your daily to-do list...,Help me set up my daily to-do list! Setting up...,97.0,0.984718,12.805576,15.310144,1.797941,2.728516,1.517578,medium,hard,medium,hard,medium
1,```python\nshopping_list = {}\nwhile True:\n ...,Create a shopping list based on my inputs!,Creating a shopping list...,Create a shopping list based on my inputs! Cre...,109.0,0.982656,11.646426,14.298523,2.395778,2.660156,1.110352,medium,hard,hard,hard,easy
2,"```python\ntotal_time = 0\nfor i in range(1, 8...",Calculate how much time I spend on my phone pe...,Calculating weekly phone usage...,Calculate how much time I spend on my phone pe...,104.0,0.984558,11.865649,6.489057,1.376707,1.870117,1.358398,medium,medium,easy,hard,medium
3,```python\ntotal_bill = float(input('Enter the...,Help me split the bill among my friends!,Splitting the bill...,Help me split the bill among my friends! Split...,89.0,0.993508,12.669135,10.379615,1.973641,2.339844,1.185547,easy,medium,medium,hard,medium
4,```python\nmovie_list = {}\nwhile True:\n g...,Organize my movie list into genres!,Organizing your movie list...,Organize my movie list into genres! Organizing...,117.0,0.989533,13.466013,13.804574,1.899647,2.625000,1.381836,medium,hard,medium,hard,medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49621,c.execute('''CREATE TABLE IF NOT EXISTS stocks...,Execute code: c.execute('''CREATE TABLE IF NOT...,Creating a 'stocks' table...,Execute code: c.execute('''CREATE TABLE IF NOT...,96.0,0.994512,31.733334,10.359362,1.295455,2.337891,1.804688,easy,medium,easy,hard,hard
49622,"c.execute(""INSERT INTO stocks VALUES ('2023-09...","Execute code: c.execute(""INSERT INTO stocks VA...",Inserting a record into the 'stocks' table...,"Execute code: c.execute(""INSERT INTO stocks VA...",128.0,0.971106,31.694456,5.411231,0.948437,1.688477,1.780273,medium,easy,easy,medium,hard
49623,c.execute('SELECT * FROM stocks WHERE symbol=?...,Execute code: c.execute('SELECT * FROM stocks ...,Querying the 'stocks' table for records with s...,Execute code: c.execute('SELECT * FROM stocks ...,91.0,0.993973,32.264351,19.505630,1.149660,2.970703,2.583984,easy,hard,easy,hard,hard
49624,c.execute('UPDATE stocks SET qty=120 WHERE sym...,Execute code: c.execute('UPDATE stocks SET qty...,Updating the 'stocks' table to change quantity...,Execute code: c.execute('UPDATE stocks SET qty...,93.0,0.990714,32.493233,16.716667,0.955600,2.816406,2.947266,easy,hard,easy,hard,hard


ValueError: В датасете нет нужных CL-колонок: ['llm_judge_category']

## 5. Подготовить список запусков

`FORCE_RETRAIN = False` означает: если у запуска уже есть `outputs/train_runs/<run>/summary.json` и LoRA-адаптер в `models/<selected_model>-<run>`, он будет пропущен. Это удобно после падения Slurm-задачи: можно просто перезапустить ноутбук/скрипт и добрать недостающие эксперименты.


In [ ]:
def run_paths(run_name):
    run_dir = Path(config.OUTPUTS_DIR) / "train_runs" / run_name
    adapter_dir = Path(config.MODELS_DIR) / f"{config.SELECTED_MODEL}-{run_name}"
    return run_dir, adapter_dir

def run_is_completed(run_name):
    run_dir, adapter_dir = run_paths(run_name)
    return (run_dir / "summary.json").exists() and adapter_dir.exists()

if RUN_MODE == "smoke":
    active_cfg = deepcopy(train_cfg)
    active_cfg["dataset"]["limit"] = 2000
    active_cfg["dataset"]["val_size"] = 100
    active_cfg["training"]["max_steps"] = 10
    active_cfg["training"]["stage_max_steps"] = 10
    active_cfg["training"]["eval_steps"] = 10
    active_cfg["training"]["save_steps"] = 10
    active_cfg["training"]["logging_steps"] = 1
    active_cfg["training"]["report_to"] = "none"
    selected_runs = [runs_by_name["cl_length"]]
else:
    active_cfg = deepcopy(train_cfg)
    selected_runs = cl_runs

if not FORCE_RETRAIN:
    before = len(selected_runs)
    selected_runs = [run for run in selected_runs if not run_is_completed(run["name"])]
    skipped = before - len(selected_runs)
else:
    skipped = 0

plan_df = pd.DataFrame([
    {
        "name": run["name"],
        "category_col": run.get("category_col"),
        "already_completed": run_is_completed(run["name"]),
        "will_run": run in selected_runs,
        "run_dir": str(run_paths(run["name"])[0]),
        "adapter_dir": str(run_paths(run["name"])[1]),
    }
    for run in (cl_runs if RUN_MODE == "full" else [runs_by_name["cl_length"]])
])

print("RUN_MODE:", RUN_MODE)
print("FORCE_RETRAIN:", FORCE_RETRAIN)
print("skipped completed:", skipped)
print("selected runs:", [run["name"] for run in selected_runs])
display(plan_df)


## 6. Запустить обучение

Эта ячейка запускает выбранные эксперименты. Для `RUN_MODE = "full"` это пять CL-вариантов; для `RUN_MODE = "smoke"` — короткая проверка `cl_length`.

Логи каждого запуска сохраняются в:

```text
outputs/train_runs/<run_name>/metrics.csv
outputs/train_runs/<run_name>/metrics.parquet
outputs/train_runs/<run_name>/summary.json
```

Адаптеры LoRA сохраняются в:

```text
models/<selected_model>-<run_name>/
```


In [ ]:
if len(selected_runs) == 0:
    print("Нет запусков для обучения: все выбранные эксперименты уже выглядят завершёнными.")
    train_result_df = pd.DataFrame()
else:
    started_at = time.time()
    train_result_df = train_lora_experiments(
        train_cfg=active_cfg,
        runs=selected_runs,
    )
    elapsed_min = (time.time() - started_at) / 60
    print(f"elapsed_min: {elapsed_min:.2f}")
    display(train_result_df)

train_result_df


## 7. Собрать общую таблицу обучения

Здесь собираются `summary.json` и последние записи из `metrics.csv`. В таблице должны появляться `loss`, `eval_loss`, `grad_norm_manual`, `learning_rate`, `epoch` и служебные поля, если они были залогированы Trainer-ом.

Файлы сохраняются в:

```text
outputs/train_runs/summary.csv
outputs/train_runs/summary.parquet
```


In [ ]:
train_summary_df = collect_training_table()
display(train_summary_df)

summary_csv = Path(config.OUTPUTS_DIR) / "train_runs" / "summary.csv"
summary_parquet = Path(config.OUTPUTS_DIR) / "train_runs" / "summary.parquet"

print("saved:", summary_csv)
print("saved:", summary_parquet)


## 8. Собрать детальные train/eval/grad-логи в одну pandas-таблицу

Эта таблица нужна, чтобы потом нормально смотреть динамику loss, eval loss и gradient norm по всем экспериментам сразу.


In [ ]:
def read_metrics_for_runs(run_names):
    frames = []
    for run_name in run_names:
        run_dir, _ = run_paths(run_name)
        parquet_path = run_dir / "metrics.parquet"
        csv_path = run_dir / "metrics.csv"

        if parquet_path.exists():
            m = pd.read_parquet(parquet_path)
        elif csv_path.exists():
            m = pd.read_csv(csv_path)
        else:
            print(f"Нет metrics для {run_name}: {run_dir}")
            continue

        if "experiment" not in m.columns:
            m["experiment"] = run_name

        frames.append(m)

    if not frames:
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True, sort=False)

metrics_df = read_metrics_for_runs(CL_RUN_NAMES)

out_train_dir = Path(config.OUTPUTS_DIR) / "train_runs"
out_train_dir.mkdir(parents=True, exist_ok=True)

all_metrics_csv = out_train_dir / "all_cl_metrics.csv"
all_metrics_parquet = out_train_dir / "all_cl_metrics.parquet"

if len(metrics_df):
    metrics_df.to_csv(all_metrics_csv, index=False)
    metrics_df.to_parquet(all_metrics_parquet, index=False)

print("rows:", len(metrics_df))
print("columns:", list(metrics_df.columns))
print("saved:", all_metrics_csv)
print("saved:", all_metrics_parquet)

display(metrics_df.tail(20))


## 9. Быстрый просмотр ключевых метрик

Ячейка не обязательная, но удобная: она вытаскивает только шаги, loss, eval loss, grad norm и learning rate. Так проще понять, не взорвались ли градиенты и не развалился ли validation loss.


In [ ]:
metric_cols = [
    "experiment",
    "stage",
    "step",
    "epoch",
    "loss",
    "eval_loss",
    "grad_norm",
    "grad_norm_manual",
    "learning_rate",
]

existing_metric_cols = [col for col in metric_cols if col in metrics_df.columns]
compact_metrics_df = metrics_df[existing_metric_cols].copy() if len(metrics_df) else pd.DataFrame(columns=existing_metric_cols)

display(compact_metrics_df.tail(50))

compact_path = Path(config.OUTPUTS_DIR) / "train_runs" / "compact_cl_metrics.csv"
compact_metrics_df.to_csv(compact_path, index=False)
print("saved:", compact_path)


## 10. Подготовить список LoRA-адаптеров для inference/eval

Для evaluation берутся только реально существующие адаптеры. Если какой-то запуск ещё не дообучен, он будет показан в `missing_adapters_df`.


In [ ]:
eval_experiments = []

if INCLUDE_BASELINE_IN_EVAL:
    eval_experiments.append((config.SELECTED_MODEL, config.MODEL_PATH, None))

missing_adapters = []

for run_name in CL_RUN_NAMES:
    _, adapter_dir = run_paths(run_name)
    if adapter_dir.exists():
        eval_experiments.append((run_name, config.MODEL_PATH, str(adapter_dir)))
    else:
        missing_adapters.append({
            "experiment": run_name,
            "expected_adapter_dir": str(adapter_dir),
        })

experiments_df = pd.DataFrame(
    eval_experiments,
    columns=["experiment", "model_path", "adapter_path"],
)

missing_adapters_df = pd.DataFrame(missing_adapters)

display(experiments_df)
display(missing_adapters_df)

if len(eval_experiments) == 0:
    print("Нет адаптеров для evaluation.")


## 11. Запустить inference/eval и сохранить pandas-таблицу

`evaluate_lora_experiments` прогоняет MBPP sanitized и HumanEval, затем сохраняет агрегированную таблицу в:

```text
outputs/eval/<out_name>.csv
outputs/eval/<out_name>.parquet
```

Для быстрого теста используется `EVAL_MAX_TASKS = 5`. Для полного eval поставь `EVAL_MAX_TASKS = None`.


In [ ]:
if len(eval_experiments) == 0:
    eval_df = pd.DataFrame()
else:
    out_name = "cl_eval_smoke" if EVAL_MAX_TASKS else "cl_eval_full"
    eval_df = evaluate_lora_experiments(
        eval_experiments,
        max_tasks=EVAL_MAX_TASKS,
        out_name=out_name,
    )
    display(eval_df)

    eval_dir = Path(config.OUTPUTS_DIR) / "eval"
    print("saved:", eval_dir / f"{out_name}.csv")
    print("saved:", eval_dir / f"{out_name}.parquet")

eval_df


## 12. Финальная сводная таблица train + eval

Эта таблица удобна для отчёта: в ней каждая строка — эксперимент, рядом последние train-метрики и eval-метрики по benchmark-ам.


In [ ]:
def make_final_summary(train_summary_df, eval_df):
    train_part = train_summary_df.copy()

    if len(eval_df):
        id_cols = ["experiment", "benchmark"]
        value_cols = [col for col in eval_df.columns if col not in id_cols + ["model_path", "adapter_path"]]

        eval_wide = eval_df.pivot_table(
            index="experiment",
            columns="benchmark",
            values=value_cols,
            aggfunc="first",
        )
        eval_wide.columns = [f"{metric}_{benchmark}" for metric, benchmark in eval_wide.columns]
        eval_wide = eval_wide.reset_index()
    else:
        eval_wide = pd.DataFrame(columns=["experiment"])

    if "experiment" in train_part.columns:
        return train_part.merge(eval_wide, on="experiment", how="outer")

    return eval_wide

final_summary_df = make_final_summary(train_summary_df, eval_df)

final_dir = Path(config.OUTPUTS_DIR) / "eval"
final_dir.mkdir(parents=True, exist_ok=True)

final_csv = final_dir / "cl_train_eval_summary.csv"
final_parquet = final_dir / "cl_train_eval_summary.parquet"

final_summary_df.to_csv(final_csv, index=False)
final_summary_df.to_parquet(final_parquet, index=False)

display(final_summary_df)

print("saved:", final_csv)
print("saved:", final_parquet)


## 13. Мини-чеклист после запуска

После полного запуска должны существовать:

```text
models/4b-thinking-cl_length/
models/4b-thinking-cl_perplexity/
models/4b-thinking-cl_lexical/
models/4b-thinking-cl_semantic/
models/4b-thinking-cl_llm_judge/

outputs/train_runs/all_cl_metrics.csv
outputs/train_runs/all_cl_metrics.parquet
outputs/train_runs/summary.csv
outputs/train_runs/summary.parquet

outputs/eval/cl_eval_full.csv
outputs/eval/cl_eval_full.parquet
outputs/eval/cl_train_eval_summary.csv
outputs/eval/cl_train_eval_summary.parquet
```

Если используешь `SELECTED_MODEL = "4b-instruct"`, префикс папок будет `4b-instruct-*`.


In [ ]:
expected_paths = []

for run_name in CL_RUN_NAMES:
    _, adapter_dir = run_paths(run_name)
    expected_paths.append(adapter_dir)
    expected_paths.append(Path(config.OUTPUTS_DIR) / "train_runs" / run_name / "metrics.csv")
    expected_paths.append(Path(config.OUTPUTS_DIR) / "train_runs" / run_name / "summary.json")

expected_paths.extend([
    Path(config.OUTPUTS_DIR) / "train_runs" / "all_cl_metrics.csv",
    Path(config.OUTPUTS_DIR) / "train_runs" / "summary.csv",
    Path(config.OUTPUTS_DIR) / "eval" / "cl_train_eval_summary.csv",
])

check_df = pd.DataFrame([
    {"path": str(path), "exists": path.exists()}
    for path in expected_paths
])

display(check_df)
